# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, getpass
import duckdb

# Colab: use the Secrets panel (key icon) to set HF_TOKEN, then userdata.get() picks it up.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Point straight at the March partition -- this is a mid-panel month, not the
# sealed final month (2026-06) and not the _sample table (that IS the final month).
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# DESCRIBE touches Parquet metadata/schema only -- near-free, and it tells us the
# real column names before we write a single WHERE clause against them.
con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_march']}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**In plain words:**

1. **One row means:** one content item's daily search-performance record — the triple
   `(report_date, client_hash_id, content_hash_id)` inside `fact_content_daily_performance`.
   My lane (Refresh / Content Opportunity Scoring) rolls this daily grain up to **one row
   per content item, summarized over one month** for scoring — but the raw table's grain,
   which I verify below, is the daily one.
2. **Table(s) I use:** `fact_content_daily_performance` (the `month=2026-03` partition only)
   for the time series; `dim_content` for content-level context (content age); `dim_clients`
   only to confirm which clients have GA4 coverage in this window.
3. **Time window:** calendar month **2026-03-01 through 2026-03-31** — a mid-panel month.
   I never touch `fact_content_daily_performance_sample` or the `2026-06` partition for label
   logic, since the sample table *is* the final month and would silently seal in the outcome
   window.
4. **What I'd predict/rank (label or proxy):** a proxy label, `is_declining_proxy` — 1 when a
   content item's GSC impressions in the second half of March (16th–31st) fall below 80% of
   its first-half impressions (1st–15th), else 0. This mirrors the starter dataset's
   `trend_direction` idea, but built myself from raw daily rows instead of trusting a
   pre-computed column. Used to rank content items into a review queue.
5. **What I deliberately exclude:** FlyRank's own product-decision fields (`health_score`,
   `priority_score`, `action_type`, `refresh_tier`) — they aren't shipped in this release, and
   I exclude them on principle: feeding a product decision into the model as a feature or
   label would just teach it to reproduce that decision, not discover anything (a circular
   result). I also exclude `keyword_hash_id` / `url_hash_id` as anything other than grouping
   keys — they're pseudonyms, not signal.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:

import pandas as pd

field_map = pd.DataFrame([
    ("report_date",       "context",  "defines the time window / the h1-vs-h2 split; never a model input"),
    ("client_hash_id",    "context",  "grouping/joining only -- a pseudonym, carries no signal"),
    ("content_hash_id",   "context",  "grouping/joining only -- a pseudonym, carries no signal"),
    ("gsc_impressions",   "feature",  "observed daily signal, known the moment the day closes"),
    ("gsc_clicks",        "feature",  "observed daily signal, known the moment the day closes"),
    ("gsc_avg_position",  "feature",  "observed daily signal, known the moment the day closes"),
    ("ga4_data_available","context",  "an availability flag used to filter rows, not to predict from"),
    ("is_declining_proxy","label",    "the thing I predict -- built from h1 vs h2 impressions, never a feature"),
    ("content_created_at","excluded", "from dim_content; excluded here -- content-age effects are a fine follow-up but out of scope for this notebook's 5-feature limit"),
    ("health_score / priority_score / action_type / refresh_tier", "excluded", "FlyRank product decisions, not shipped, and circular if ever rebuilt and reused as a label"),
])
field_map.columns = ["field", "bucket", "why"]
field_map

,field,bucket,why
0,report_date,context,defines the time window / the h1-vs-h2 split; ...
1,client_hash_id,context,"grouping/joining only -- a pseudonym, carries ..."
2,content_hash_id,context,"grouping/joining only -- a pseudonym, carries ..."
3,gsc_impressions,feature,"observed daily signal, known the moment the da..."
4,gsc_clicks,feature,"observed daily signal, known the moment the da..."
5,gsc_avg_position,feature,"observed daily signal, known the moment the da..."
6,ga4_data_available,context,"an availability flag used to filter rows, not ..."
7,is_declining_proxy,label,the thing I predict -- built from h1 vs h2 imp...
8,content_created_at,excluded,from dim_content; excluded here -- content-age...
9,health_score / priority_score / action_type / ...,excluded,"FlyRank product decisions, not shipped, and ci..."


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 of 3 — grain: is one row really `(report_date, client_hash_id, content_hash_id)`?

In [10]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"duplicate-grain rows found: {len(grain_check)} (0 means the grain holds)")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows found: 0 (0 means the grain holds)


,report_date,client_hash_id,content_hash_id,n


### Query 2 of 3 — my slice's row count and date span

In [11]:
span_check = con.sql(f"""
    SELECT
        COUNT(*)                      AS total_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        MIN(report_date)              AS min_date,
        MAX(report_date)              AS max_date
    FROM {TABLES['fact_march']}
""").df()

span_check

,total_rows,distinct_content_items,distinct_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### Query 3 of 3 — availability: filter with `IS TRUE`, how many rows survive?

Rows before a client's GA4 start are zero-filled, not zero-engagement — the flag is what
tells the two apart.

In [12]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*)                                            AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_march']}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.